# 회계기준 질의회신 특화 LLM: LoRA 파인튜닝 + RAG 하이브리드

한국회계기준원·금융감독원 질의회신 사례(KIFRS.com)를 활용해

1. **LoRA 파인튜닝**: 회계기준을 인용하며 답변하는 자문 스타일 학습
2. **RAG**: 유사 질의회신 사례 검색 후 근거로 제공
3. **평가**: Base / RAG / LoRA / LoRA+RAG 4개 구성 비교 (홀드아웃 30문항)

**환경**: Colab A100 80GB 기준. 런타임 → 런타임 유형 변경 → A100 GPU 선택.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q transformers==4.46.3 trl==0.12.1 peft==0.13.2 accelerate datasets faiss-cpu sentence-transformers==3.3.1 requests

## 1. 데이터 수집 및 전처리
저장소를 클론하고 스크래퍼를 직접 실행한다 (원본 데이터는 저작권 문제로 저장소에 포함하지 않음).

In [ ]:
REPO_URL = "https://github.com/gyeongmin100/kifrs-qa-llm.git"  # 본인 저장소로 변경
!git clone {REPO_URL} repo
%cd repo
!python src/scrape.py --rows 200 --sleep 0.3
!python src/prepare_data.py

In [ ]:
import json

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]

train_data = load_jsonl("data/processed/train.jsonl")
test_data = load_jsonl("data/processed/test.jsonl")
rag_corpus = load_jsonl("data/processed/rag_corpus.jsonl")
print(f"train {len(train_data)} / test {len(test_data)} / rag {len(rag_corpus)}")
print(train_data[0]["title"])
print(train_data[0]["question"][:200])
print(train_data[0]["answer"][:200])

## 2. LoRA 파인튜닝 (Qwen2.5-7B-Instruct, bf16)
A100 80GB이므로 양자화 없이 bf16 LoRA로 학습한다. 14B로 바꾸려면 `MODEL_ID`만 수정.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
SYSTEM_PROMPT = (
    "당신은 한국 회계기준(K-IFRS, 일반기업회계기준) 전문 회계 자문가입니다. "
    "질의된 회계처리 사안에 대해 관련 회계기준과 문단을 인용하며 명확한 결론을 제시하세요."
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto"
)

In [ ]:
from datasets import Dataset

def to_chat(example):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["question"]},
        {"role": "assistant", "content": example["answer"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

ds = Dataset.from_list(train_data).map(to_chat, remove_columns=Dataset.from_list(train_data).column_names)
print(ds[0]["text"][:500])

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

sft_config = SFTConfig(
    output_dir="outputs/lora-kifrs",
    num_train_epochs=2,
    per_device_train_batch_size=4,   # A100 80GB + checkpointing 기준 ~47GB
    gradient_accumulation_steps=4,   # 유효 배치 16
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    max_seq_length=2048,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(model=model, args=sft_config, train_dataset=ds, peft_config=peft_config)
trainer.train()

In [ ]:
ADAPTER_DIR = "outputs/lora-kifrs/final"
trainer.save_model(ADAPTER_DIR)
print("어댑터 저장:", ADAPTER_DIR)

## 3. RAG 인덱스 구축 (bge-m3 + FAISS)
질의회신 사례의 제목+질문을 임베딩하여 유사 사례를 검색한다. 홀드아웃 테스트 문서는 인덱스에서 제외되어 있음(데이터 누출 방지).

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# bge-m3 임베딩 (FlagEmbedding은 transformers 4.46과 버전 충돌이 있어 sentence-transformers 사용)
embedder = SentenceTransformer("BAAI/bge-m3", model_kwargs={"torch_dtype": torch.float16})

corpus_texts = [f"{r['title']}\n{r['question']}" for r in rag_corpus]
emb = embedder.encode(corpus_texts, batch_size=64, convert_to_numpy=True, show_progress_bar=True).astype("float32")
faiss.normalize_L2(emb)
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print("인덱스 크기:", index.ntotal)

In [ ]:
def retrieve(question, k=3):
    q_emb = embedder.encode([question], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, k)
    return [(rag_corpus[i], float(s)) for i, s in zip(idxs[0], scores[0])]

# 리트리벌 확인
for doc, score in retrieve(test_data[0]["question"]):
    print(f"{score:.3f}", doc["docNumber"], doc["title"][:50])

## 4. 생성 파이프라인 (4가지 구성)

In [ ]:
RAG_TEMPLATE = (
    "다음은 과거 유사한 질의회신 사례입니다.\n\n{context}\n\n"
    "위 사례를 참고하여 아래 질의에 답변하세요.\n\n[질의]\n{question}"
)

def build_prompt(question, use_rag=False):
    if not use_rag:
        return question
    docs = retrieve(question, k=3)
    context = "\n\n".join(
        f"[사례 {i+1}] {d['title']}\n질의: {d['question'][:500]}\n회신: {d['answer'][:500]}"
        for i, (d, _) in enumerate(docs)
    )
    return RAG_TEMPLATE.format(context=context, question=question)

@torch.no_grad()
def generate(model, question, use_rag=False, max_new_tokens=512):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_prompt(question, use_rag)},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

In [ ]:
# 학습 전(Base) 모델과 학습 후(LoRA) 모델 준비
# trainer가 model에 어댑터를 부착한 상태이므로:
#  - LoRA 답변: 어댑터 활성화 상태로 생성
#  - Base 답변: disable_adapter()로 어댑터 비활성화 후 생성
lora_model = trainer.model
lora_model.eval()

def generate_config(question, config):
    """config: 'base' | 'rag' | 'lora' | 'lora_rag'"""
    use_rag = config in ("rag", "lora_rag")
    if config in ("base", "rag"):
        with lora_model.disable_adapter():
            return generate(lora_model, question, use_rag)
    return generate(lora_model, question, use_rag)

In [ ]:
# 파인튜닝 전후 스타일 비교 (테스트 1문항)
q = test_data[0]["question"]
print("=== 질의 ===", q[:300], sep="\n")
print("\n=== Base ===", generate_config(q, "base")[:600], sep="\n")
print("\n=== LoRA ===", generate_config(q, "lora")[:600], sep="\n")
print("\n=== 정답 ===", test_data[0]["answer"][:600], sep="\n")

## 5. 평가: 홀드아웃 30문항 × 4구성
정답(실제 회신)과 모델 답변의 임베딩 유사도를 자동 지표로 사용하고, 전체 답변을 저장해 정성 평가한다.

In [ ]:
import os
from tqdm.auto import tqdm

os.makedirs("results", exist_ok=True)

CONFIGS = ["base", "rag", "lora", "lora_rag"]
results = []
for rec in tqdm(test_data):
    row = {"docNumber": rec["docNumber"], "title": rec["title"],
           "question": rec["question"], "reference": rec["answer"]}
    for cfg in CONFIGS:
        row[cfg] = generate_config(rec["question"], cfg)
    results.append(row)

with open("results/eval_answers.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print("저장: results/eval_answers.json")

In [ ]:
import pandas as pd

def embed_sim(texts_a, texts_b):
    ea = embedder.encode(texts_a, convert_to_numpy=True)
    eb = embedder.encode(texts_b, convert_to_numpy=True)
    ea = ea / np.linalg.norm(ea, axis=1, keepdims=True)
    eb = eb / np.linalg.norm(eb, axis=1, keepdims=True)
    return (ea * eb).sum(axis=1)

refs = [r["reference"] for r in results]
scores = {cfg: embed_sim([r[cfg] for r in results], refs) for cfg in CONFIGS}
df = pd.DataFrame(scores, index=[r["docNumber"] for r in results])
summary = df.agg(["mean", "std"]).T
summary.columns = ["정답 유사도(평균)", "표준편차"]
print(summary.round(4))
df.to_csv("results/eval_similarity.csv", encoding="utf-8-sig")
summary.to_csv("results/eval_summary.csv", encoding="utf-8-sig")

## 6. 결과물 저장 (Google Drive)
LoRA 어댑터와 평가 결과를 Drive에 백업한다. 어댑터는 GitHub에 직접 커밋해도 됨(수백 MB 이하).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r outputs/lora-kifrs/final /content/drive/MyDrive/lora-kifrs-adapter
!cp -r results /content/drive/MyDrive/lora-kifrs-results
print("Drive 백업 완료")

## 7. 실시간 채팅 데모 (Gradio)
실행하면 공개 URL(`https://xxx.gradio.live`)이 출력된다. 브라우저에서 열어 채팅으로 질의 가능.
드롭다운으로 4가지 구성(base/rag/lora/lora_rag)을 전환하며 비교할 수 있다.

학습(2번 섹션)과 RAG 인덱스(3번 섹션) 셀을 먼저 실행해야 함. 단일 턴 질의응답 방식(대화 맥락 미반영).

In [ ]:
!pip install -q "gradio==4.44.1"  # 최신 gradio 5.x는 transformers 4.46이 잡아둔 구버전 huggingface_hub와 충돌

In [ ]:
import gradio as gr

def chat_fn(message, history, config):
    answer = generate_config(message, config)
    if config in ("rag", "lora_rag"):
        docs = retrieve(message, k=3)
        refs = "\n".join(
            f"- [{d['docNumber']}] {d['title']} (유사도 {s:.2f})" for d, s in docs
        )
        answer += f"\n\n---\n**참고한 질의회신 사례**\n{refs}"
    return answer

demo = gr.ChatInterface(
    chat_fn,
    additional_inputs=[
        gr.Dropdown(["lora_rag", "lora", "rag", "base"], value="lora_rag", label="모델 구성")
    ],
    title="회계기준 질의회신 어시스턴트",
    description="K-IFRS·일반기업회계기준 질의를 입력하세요. LoRA 파인튜닝 + 유사 사례 RAG 기반으로 답변합니다.",
    examples=[
        ["회사가 보유한 자기주식을 소각하는 경우 회계처리는 어떻게 하나요?"],
        ["리스기간이 12개월 이하인 사무실 임차 계약도 사용권자산을 인식해야 하나요?"],
    ],
)
demo.launch(share=True)